# Experiment: Private Send Readiness Gate

Objective: classify whether the public evidence trail can move to a later private execution check without contacting anyone or leaking private details.


In [ ]:
from __future__ import annotations

READY = "private_send_ready_for_private_execution_check"
HOLD_APPROVAL = "private_send_hold_missing_scoped_approval"
HOLD_RECIPIENT = "private_send_hold_missing_recipient_qualification"
HOLD_PACKET = "private_send_hold_missing_packet_pass"
HOLD_RESPONSE = "private_send_hold_missing_response_capture"
REJECT = "private_send_reject_public_or_clinical_scope"

SCOPED_APPROVAL = "approval_scope_public_safe_quote_only"
BLOCKERS = {
    "raw_private_content",
    "patient_records",
    "patient_samples",
    "treatment_or_dosing",
    "trial_screening",
    "purchase_scope",
    "import_scope",
    "procurement_scope",
    "ambiguous_scope",
}


def classify_private_send_readiness(record: dict[str, bool | str]) -> str:
    """Return the public-safe readiness label for a private send workflow."""
    if any(record.get(flag, False) for flag in BLOCKERS):
        return REJECT
    if record.get("approval_label") != SCOPED_APPROVAL:
        return HOLD_APPROVAL
    if not record.get("recipient_qualified", False):
        return HOLD_RECIPIENT
    if not record.get("outbound_packet_gate_passed", False):
        return HOLD_PACKET
    if not record.get("response_capture_ready", False):
        return HOLD_RESPONSE
    if not record.get("sender_recipient_private", False):
        return REJECT
    return READY


base_ready = {
    "approval_label": SCOPED_APPROVAL,
    "recipient_qualified": True,
    "outbound_packet_gate_passed": True,
    "response_capture_ready": True,
    "sender_recipient_private": True,
}

cases = {
    "missing_approval": {},
    "missing_recipient": {**base_ready, "recipient_qualified": False},
    "missing_packet": {**base_ready, "outbound_packet_gate_passed": False},
    "missing_response": {**base_ready, "response_capture_ready": False},
    "treatment_scope": {**base_ready, "treatment_or_dosing": True},
    "procurement_scope": {**base_ready, "procurement_scope": True},
    "ready_all_gates": base_ready,
}

expected = {
    "missing_approval": HOLD_APPROVAL,
    "missing_recipient": HOLD_RECIPIENT,
    "missing_packet": HOLD_PACKET,
    "missing_response": HOLD_RESPONSE,
    "treatment_scope": REJECT,
    "procurement_scope": REJECT,
    "ready_all_gates": READY,
}

results = {
    name: classify_private_send_readiness(record) for name, record in cases.items()
}
assert results == expected
current_case_state = classify_private_send_readiness({})
assert current_case_state == HOLD_APPROVAL

decision_summary = {
    "current_case_state": current_case_state,
    "ready_label": READY,
    "boundary": "ready means private execution check only, not contact permission",
}
decision_summary

## Decision

Keep the workflow on hold. Approval and recipient qualification are still missing, and the public repo must never contain private sender or recipient details.
